Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import torch
print(torch.cuda.get_device_name(0))
print("显存总量:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
print("当前已用:", torch.cuda.memory_allocated() / 1e9, "GB")

Tesla T4
显存总量: 15.637086208 GB
当前已用: 0.0 GB


Install independencies

In [3]:
!pip install "git+https://github.com/huggingface/transformers.git" --break-system-packages
!pip install accelerate bitsandbytes qwen-vl-utils --break-system-packages

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-qs5qtr_z
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-qs5qtr_z
  Resolved https://github.com/huggingface/transformers.git to commit 74d576be116be1cd356b920705365e81641d17ee
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.7.0.dev0-py3-none-any.whl size=11612715 sha256=7932ff91f4caf2e3833376b8f6194392dffa815389c02da7a321757ee76bcf95
  Stored in directory: /tmp/pip-ephem-wheel-cache-iksqpsja/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Load the model

In [4]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_name = "Qwen/Qwen3-VL-4B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading Qwen3-VL-4B...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_name)
model.eval()
print("Qwen3-VL-4B loaded!")

Loading Qwen3-VL-4B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Qwen3-VL-4B loaded!


For one image

In [ ]:
from PIL import Image
import torch
import re

image_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images/31.jpg"
image = Image.open(image_path).convert("RGB")

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt_text}
        ]
    }
]

text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)

gen = outputs[0][inputs["input_ids"].shape[-1]:]
raw = processor.decode(gen, skip_special_tokens=True).strip()

print("RAW OUTPUT:", raw)

m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
label = m.group(1) if m else None
print("PARSED LABEL:", label)

RAW OUTPUT: Class labels: Non_LGBT

Thought: The meme uses the term "同志" (comrade) and "同志運動" (comrade movement) in a satirical or humorous context, likely referencing political or social movements. The image features a political figure and two edited figures in athletic poses, which is commonly used in internet memes to create absurd or ironic comparisons. There is no explicit or implicit reference to LGBTQ+ identities, and the meme does not contain content that would be considered harmful in the context of homophobia, transphobia, or any other discriminatory category. The meme is satirical and does not target any protected group under the specified categories.
PARSED LABEL: Non_LGBT


For multi images

删除错误结果

In [5]:
import os
import json
import re
import torch
from PIL import Image
from tqdm import tqdm

image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
output_json = "/content/drive/MyDrive/Qwen3VL4B_HM_ZeroShot_pred.json"

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

image_files = sorted(
    [f for f in os.listdir(image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(image_dir, img_name)

    try:
        image = Image.open(img_path).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text}
                ]
            }
        ]

        text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)

        gen = outputs[0][inputs["input_ids"].shape[-1]:]
        raw = processor.decode(gen, skip_special_tokens=True).strip()

        m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
        if m:
            label = m.group(1)
        elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
            label = "Non_LGBT"
        else:
            label = "UNKNOWN"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

发现已有成功结果 184 条，从断点继续...
剩余待处理: 48 张


推理进度:   2%|▏         | 1/48 [00:29<23:19, 29.77s/it]

✅ 191.jpg -> Non_LGBT


推理进度:   4%|▍         | 2/48 [01:53<47:02, 61.36s/it]

✅ 192.jpg -> Non_LGBT


...

✅ 237.jpg -> Non_LGBT


推理进度:  98%|█████████▊| 47/48 [38:28<00:42, 42.17s/it]

✅ 238.jpg -> Non_LGBT


推理进度: 100%|██████████| 48/48 [39:25<00:00, 49.29s/it]

✅ 239.jpg -> Non_LGBT

完成！共 232 条结果已保存
  ERROR: 2
  Homophobia: 25
  Non_LGBT: 193
  Transphobia: 7
  homophobia: 5


In [6]:
import json

with open("/content/drive/MyDrive/Qwen3VL4B_HM_ZeroShot_pred.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

labels = [p["predicted_label"] for p in predictions]
print(f"总数: {len(predictions)}")
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

总数: 232
  ERROR: 2
  Homophobia: 25
  Non_LGBT: 193
  Transphobia: 7
  homophobia: 5


In [7]:
import json

output_json = "/content/drive/MyDrive/Qwen3VL4B_HM_ZeroShot_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

# 修正大小写
for p in predictions:
    if p["predicted_label"].lower() == "homophobia":
        p["predicted_label"] = "Homophobia"
    elif p["predicted_label"].lower() == "transphobia":
        p["predicted_label"] = "Transphobia"
    elif p["predicted_label"].lower() == "non_lgbt":
        p["predicted_label"] = "Non_LGBT"

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

labels = [p["predicted_label"] for p in predictions]
print(f"总数: {len(predictions)}")
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

总数: 232
  ERROR: 2
  Homophobia: 30
  Non_LGBT: 193
  Transphobia: 7


In [8]:
import os
import torch
import re
from PIL import Image

error_items = [p for p in predictions if p["predicted_label"] == "ERROR"]
print(f"需要重跑: {len(error_items)} 张")

image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"

for p in error_items:
    img_name = p["image_name"]
    img_path = os.path.join(image_dir, img_name)
    print(f"重跑: {img_name}")

    try:
        torch.cuda.empty_cache()
        image = Image.open(img_path).convert("RGB")
        # 限制尺寸
        if max(image.size) > 800:
            image.thumbnail((800, 800))

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text}
                ]
            }
        ]

        text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)

        gen = outputs[0][inputs["input_ids"].shape[-1]:]
        raw = processor.decode(gen, skip_special_tokens=True).strip()

        m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
        label = m.group(1) if m else "UNKNOWN"

        p["predicted_label"] = label
        p["raw_output"] = raw
        print(f"✅ {img_name} -> {label}")

        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ {img_name} 还是出错: {e}")

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

labels = [p["predicted_label"] for p in predictions]
print(f"\n最终统计:")
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

需要重跑: 2 张
重跑: 211.jpg
✅ 211.jpg -> Non_LGBT
重跑: 213.jpg
✅ 213.jpg -> Non_LGBT

最终统计:
  Homophobia: 30
  Non_LGBT: 195
  Transphobia: 7


Check saved or not